In [ ]:
!pip install yfinance PyPortfolioOpt --quiet

In [ ]:
import yfinance as yf
from pypfopt import expected_returns, risk_models
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from plotly.subplots import make_subplots
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
us = [
   ("SXR8","XETR"),
   ("ZPRR","XETR")
]
anti_US = [
   ("LGQM","XETR"),
   ("IUSC","XETR"),
   "LU1681045024",#("ALAT","XBUD"),
   ("IQQF","XETR"),
   ("CEBL","XETR"),
   ("LASP","XETR"),
   ("XCS5","XETR")
  ]
bravos = [
    # 1. URÁN (Uranium choke point)
    "CCJ",   # Cameco Corp - A legnagyobb nyugati uránbányász
    "UEC",   # Uranium Energy Corp - Amerikai fókuszú, gyorsan növekvő bányász
    "NXE",   # NexGen Energy - Hatalmas kanadai lelőhely fejlesztője

    # 2. ENERGETIKAI INFRASTRUKTÚRA & AI (Power for data centers)
    "CEG",   # Constellation Energy - Atomenergia szolgáltató AI központoknak
    "VST",   # Vistra Corp - Hasonló profilú közműszolgáltató
    "MTZ",   # MasTec Inc - Infrastruktúra építés (hálózatok, adatközpontok)
    "EME",   # EMCOR Group - Mechanikai/elektromos rendszerek adatközpontokhoz

    # 3. ALAPFÉMEK / RÉZ (Building blocks of the new economy)
    "FCX",   # Freeport-McMoRan - A világ egyik legnagyobb rézbányásza
    "TECK",  # Teck Resources - Jelentős réz- és fémkitettség
    "BHP",   # BHP Group - Diverzifikált bányászat, erős réz fókusszal

    # 4. NEMESFÉMEK & FEDEZETI ESZKÖZÖK
    #"GLD",   # SPDR Gold Shares - Arany ETF (a videó központi témája)
    #"WPM"    # Wheaton Precious Metals - Alacsony költségű nemesfém streaming
]

In [ ]:
tickers = us + anti_US
budget = 10_000
base_currency = "EUR"

#time range
#start_date = "2020-01-01"
#end_date = "2024-01-01"
lookback_months = 12*1
end_date = pd.Timestamp.today()#pd.Timestamp("2026-03-10")
start_date = end_date - pd.DateOffset(months=lookback_months)

In [ ]:
# 1.Download data
stock_currencies = {}
fx_tickers_needed = set()

raw_data = pd.DataFrame()
names = dict()

for ticker in tickers:
    T = yf.Ticker(ticker)
    history = T.history(start=start_date, end=end_date, repair=True)
    history.index = history.index.tz_localize(None)
    raw_data[ticker] = history['Close']

    stock_info = T.info
    currency = stock_info.get('currency', 'USD')
    stock_currencies[ticker] = currency
    names[ticker] = stock_info.get('shortName', ticker if isinstance(ticker, str) else ticker[0])
    #remove duplicate space from name
    while '  ' in names[ticker]:
        names[ticker] = names[ticker].replace('  ', ' ')

    if currency != base_currency:
        # Yahoo Finance formats FX pairs like [FROM][TO]=X
        fx_ticker = f"{currency}{base_currency}=X"
        fx_tickers_needed.add(fx_ticker)

In [ ]:
if fx_tickers_needed:
    print(f"Downloading FX data for: {fx_tickers_needed}")
    fx_data = yf.download(fx_tickers_needed, start=start_date, end=end_date, auto_adjust=True).xs('Close', level=0, axis=1)
    fx_data.index = fx_data.index.tz_localize(None)
    raw_data = pd.concat([raw_data, fx_data], axis=1)

# Clean up the data (forward-fill missing holiday data, drop remaining NaNs)
raw_data = raw_data.ffill().dropna()

# Do the automatic conversion math
df = pd.DataFrame(index=raw_data.index)

for ticker in tickers:
    currency = stock_currencies[ticker]

    if currency == base_currency:
        # No conversion needed, just copy it over
        df[names[ticker]] = raw_data[ticker]
    else:
        # Multiply the stock price by the exchange rate
        # e.g., Stock in USD * (USDEUR=X) = Stock in EUR
        fx_ticker = f"{currency}{base_currency}=X"
        df[names[ticker]] = raw_data[ticker] * raw_data[fx_ticker]

In [ ]:
# 2. Calculate returns and covariance
mu = expected_returns.mean_historical_return(df)
S = risk_models.sample_cov(df)

# 3. Optimize for Maximum Sharpe Ratio
ef = EfficientFrontier(mu, S)
weights = ef.max_sharpe()
cleaned_weights = ef.clean_weights()

# Get performance metrics for later use
ret_opt, std_opt, sharpe_opt = ef.portfolio_performance(verbose=True)

# 4. Discrete Allocation
latest_prices = get_latest_prices(df)
da = DiscreteAllocation(cleaned_weights, latest_prices, total_portfolio_value=budget)
allocation, leftover = da.lp_portfolio()

print(f"\nShares to buy: {allocation}")
print(f"Cash leftover: {leftover:.2f} {base_currency}")

In [ ]:
# 1. Generate random portfolios (the cloud)
n_samples = 20000
random_weights = np.random.dirichlet(np.full(len(tickers), 0.3), n_samples)
random_returns = random_weights.dot(mu)
random_stds = np.array([np.sqrt(np.dot(w.T, np.dot(S, w))) for w in random_weights])
random_sharpes = random_returns / random_stds

# 2. Individual stock metrics
asset_stds = np.sqrt(np.diag(S))

# 3. Calculate frontier curve
# Find the minimum volatility portfolio to start the frontier curve
ef_min_vol = EfficientFrontier(mu, S)
ef_min_vol.min_volatility()
min_vol_ret, _, _ = ef_min_vol.portfolio_performance()

# Generate points along the efficient frontier
frontier_returns = np.linspace(min_vol_ret, mu.max())
frontier_vols = []
valid_returns = []

for r in frontier_returns:
    try:
        ef_line = EfficientFrontier(mu, S)
        ef_line.efficient_return(target_return=r)
        _, vol, _ = ef_line.portfolio_performance()
        frontier_vols.append(vol)
        valid_returns.append(r)
    except:
        pass # Skip if the optimizer fails to converge for a specific point


# 4. Build Plotly figure
fig = go.Figure()

# Add Random Mixes
fig.add_trace(go.Scatter(
    x=random_stds, y=random_returns, mode='markers',
    marker=dict(color=random_sharpes, colorscale='Viridis', showscale=True, colorbar=dict(title='Sharpe Ratio'), size=4, opacity=0.6),
    name='Random Mixes',
    text=[f'Return: {r*100:.2f}%<br>Risk: {v*100:.2f}%<br>Sharpe: {s:.2f} <br>Weights:{[f'{ww*100:.2f}%' for ww in w]}'
           for r, v, s, w in zip(random_returns, random_stds, random_sharpes, random_weights)],
    hoverinfo='text'
))

# Add Individual Stocks
display_names = [names[t] for t in tickers]
fig.add_trace(go.Scatter(
    x=asset_stds, y=mu, mode='markers+text',
    text=display_names, textposition='middle right',
    marker=dict(color='red', size=8, symbol='x-thin', line=dict(width=2)),
    name='Individual Stocks',
    hovertext=[f'{t}<br>Return: {r*100:.2f}%<br>Risk: {v*100:.2f}%' for t, r, v in zip(display_names, mu, asset_stds)],
    hoverinfo='text'
))

# Add the Optimal Portfolio (using previously calculated ret_opt, std_opt)
fig.add_trace(go.Scatter(
    x=[std_opt], y=[ret_opt], mode='markers',
    marker=dict(color='gold', size=16, symbol='star', line=dict(color='black', width=1)),
    name='Optimal Portfolio',
    hovertext=[f'MAX SHARPE<br>Return: {ret_opt*100:.2f}%<br>Risk: {std_opt*100:.2f}%'],
    hoverinfo='text'
))

# Add the Exact Efficient Frontier Line (Non-clickable) ---
fig.add_trace(go.Scatter(
    x=frontier_vols, y=valid_returns,
    mode='lines',
    line_shape='spline',
    line=dict(color='red', width=3, dash='dot'),
    name='Efficient Frontier',
    hoverinfo='skip' # This makes it ignore mouse hovers (non-clickable)
))

fig.update_layout(
    title='Efficient Frontier',
    xaxis_title='Risk (Expected Annual Volatility)',
    yaxis_title='Expected Annual Return',
    template='plotly_dark',
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
    yaxis=dict(tickformat=".0%"),
    xaxis=dict(tickformat=".0%"),
    height=750

)

fig.show()

In [ ]:
out = widgets.Output()

def update_portfolio(target_return):
    with out:
        try:
            # 1. Optimize for target return
            ef = EfficientFrontier(mu, S)
            ef.efficient_return(target_return / 100)
            raw_weights = ef.clean_weights()
            ret, vol, sharpe = ef.portfolio_performance(verbose=False)

            # 2. Discrete Allocation (Whole Shares)
            latest_prices = get_latest_prices(df)
            da = DiscreteAllocation(raw_weights, latest_prices, total_portfolio_value=budget)
            allocation, leftover = da.greedy_portfolio()

            # 3. Data Preparation
            df_plot = pd.DataFrame.from_dict(raw_weights, orient='index', columns=['Ideal_Weight'])
            df_plot['Shares'] = pd.Series(allocation).fillna(0)
            df_plot['Actual_Value'] = df_plot['Shares'] * latest_prices
            df_plot['Actual_Weight'] = df_plot['Actual_Value'] / budget

            # Filter to show only tickers with any allocation
            df_plot = df_plot[(df_plot['Ideal_Weight'] > 0.0001) | (df_plot['Actual_Weight'] > 0.0001)]
            df_plot = df_plot.sort_values(by='Ideal_Weight', ascending=False)

            # 4. Plotting with Subplots
            fig = make_subplots(
                rows=1, cols=2,
                column_widths=[0.75, 0.25],
                horizontal_spacing=0.05, # Tightened spacing for the wider screen
                subplot_titles=("Portfolio Composition", "Risk/Return Profile")
            )

            # TRACE 1: IDEAL (The "Ghost" Bar)
            fig.add_trace(go.Bar(
                x=df_plot.index,
                y=df_plot['Ideal_Weight'],
                name='Target Weight',
                marker=dict(
                    color='rgba(255, 255, 255, 0.1)',
                    line=dict(color='rgba(255, 255, 255, 0.3)', width=1)
                ),
                width=0.8,
                hoverinfo='skip'
            ), row=1, col=1)

            # TRACE 2: ACTUAL (The Solid Bar)
            fig.add_trace(go.Bar(
                x=df_plot.index,
                y=df_plot['Actual_Weight'],
                name='Actual Weight',
                marker_color='teal',
                width=0.4,
                customdata=df_plot[['Shares', 'Actual_Value', 'Ideal_Weight']],
                hovertemplate=(
                    "<b>%{x}</b><br>" +
                    "Shares: %{customdata[0]}<br>" +
                    "Cost: %{customdata[1]:.2f} " + f"{base_currency}<br>" +
                    "Actual Weight: %{y:.2%}<br>" +
                    "Target Weight: %{customdata[2]:.2%}" +
                    "<extra></extra>"
                )
            ), row=1, col=1)

            # TRACE 3: EFFICIENT FRONTIER CURVE
            fig.add_trace(go.Scatter(
                x=frontier_vols,
                y=valid_returns,
                mode='lines',
                line=dict(color='rgba(255, 255, 255, 0.5)', width=2, dash='dash'),
                name='Efficient Frontier',
                showlegend=False,
                hoverinfo='skip'
            ), row=1, col=2)

            # TRACE 4: SELECTED PORTFOLIO POINT
            fig.add_trace(go.Scatter(
                x=[vol],
                y=[ret],
                mode='markers',
                marker=dict(color='gold', size=16, symbol='star', line=dict(color='black', width=1)),
                name='Selected Portfolio',
                showlegend=False,
                hovertemplate="<b>Selected</b><br>Risk: %{x:.2%}<br>Return: %{y:.2%}<extra></extra>"
            ), row=1, col=2)

            # Update Layout
            fig.update_layout(
                barmode='overlay',
                template="plotly_dark",
                height=550,
                width=1500, # Your new width
                # Left margin set to 250px to house the sidebar content
                margin=dict(t=80, b=60, l=250, r=40),
                showlegend=True,
                legend=dict(
                    orientation="v",
                    yanchor="top", y=1.0,
                    # x=-0.18 is the "sweet spot" for a 1500px width with 250px margin
                    xanchor="left", x=-0.18
                )
            )

            # Add Fixed Data Annotation (Stats aligned under the legend)
            fig.add_annotation(
                text=f"<b>Target:</b><br>{target_return:.2f}%<br><br><b>Risk:</b><br>{vol*100:.2f}%<br><br><b>Cash Left:</b><br>{leftover:.2f} {base_currency}",
                align="left",
                showarrow=False,
                xref="paper", yref="paper",
                x=-0.18, y=0.0,
                xanchor="left", yanchor="bottom",
                font=dict(size=14, color="white")
            )

            # Update Axes
            fig.update_xaxes(title_text="Products",row=1, col=1)
            fig.update_yaxes(title_text="Weight Allocation", tickformat=".0%", range=[0, max(df_plot['Ideal_Weight'].max(), 0.1) * 1.1], row=1, col=1)

            fig.update_xaxes(title_text="Risk", tickformat=".1%", row=1, col=2)
            fig.update_yaxes(title_text="Return", tickformat=".1%", row=1, col=2)

            clear_output(wait=True)
            fig.show()

        except Exception as e:
            clear_output(wait=True)
            print(f"Error: {e}")

# Slider Setup (Wider slider to match the 1500px chart)
min_ret = max(0, mu.min() * 100)
max_ret = mu.max() * 100

slider = widgets.FloatSlider(
    value=ret_opt*100,
    min=round(min_ret, 1),
    max=round(max_ret, 1),
    step=0.1,
    description='Return(%)',
    continuous_update=False,
    layout=widgets.Layout(width='1500px') # Slightly less than 1500 to account for browser padding
)

display(widgets.VBox([slider, out]))
interactive_plot = widgets.interactive_output(update_portfolio, {'target_return': slider})

In [ ]:
# 1. Normalize the dataframe so every stock starts at exactly 1.0
normalized_df = df / df.iloc[0]

# 2. Sort the columns by their final value (highest to lowest)
# This ensures the legend order matches the final position on the right of the chart
final_values = normalized_df.iloc[-1].sort_values(ascending=False)
normalized_df = normalized_df[final_values.index]

# 3. Build the Plotly figure
fig = go.Figure()

for column in normalized_df.columns:
    fig.add_trace(go.Scatter(
        x=normalized_df.index,
        y=normalized_df[column],
        mode='lines',
        name=column,
        hovertemplate='%{y:.2f}x'
    ))

# --- NEW: Add a highlighted horizontal line at y=1 ---
fig.add_hline(
    y=1.0,
    line_dash="dash",
    line_color="white",
    line_width=2,
    opacity=0.7,
)

# 4. Format the chart
fig.update_layout(
    title='Relative Growth of Assets Over Time',
    xaxis_title='Date',
    yaxis_title='Growth Multiplier',
    template='plotly_dark',
    hovermode='x unified',
    # Move legend to the right side and stack it vertically
    legend=dict(
        title="Assets",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02 # Pushes it slightly outside the right edge of the plot area
    ),
    margin=dict(r=150, t=60), # Adds right margin so the legend doesn't get cut off
    height=500
)

fig.show()